In [16]:
import os
import pandas as pd
import numpy as np
import torch

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    NllbTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
)
import evaluate
from peft import LoraConfig, get_peft_model, TaskType

In [17]:
CSV_PATH = "full_corpus.csv"
MODEL_NAME = "facebook/nllb-200-distilled-600M"
tokenizer = NllbTokenizer.from_pretrained(MODEL_NAME, src_lang="ron_Latn", tgt_lang="rom_Latn",
                                          additional_special_tokens=["rom_Latn", "ron_Latn"])

RROMANI_ID = tokenizer.convert_tokens_to_ids("rom_Latn")
print(f"Corrected Rromani ID: {RROMANI_ID}")

Corrected Rromani ID: 256204


In [18]:
df = pd.read_csv(CSV_PATH)

# Rename if needed
df = df.rename(columns={"Text_ro": "ro", "Text_rom": "rmy"})

# Drop invalid rows
df = df.dropna(subset=["ro", "rmy"])
df = df[df["ro"].str.strip() != ""]
df = df[df["rmy"].str.strip() != ""]

df = df.reset_index(drop=True)

print("Rows after cleaning:", len(df))


Rows after cleaning: 10379


In [19]:
dataset = Dataset.from_pandas(df)

dataset = dataset.train_test_split(test_size=0.1, seed=42)
train_ds = dataset["train"]
eval_ds = dataset["test"]


In [20]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.version.cuda)

2.6.0+cu124
True
12.4


In [21]:
bleu_metric = evaluate.load("sacrebleu")
chrf_metric = evaluate.load("chrf")

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]

    # Safety check: Clip IDs to the tokenizer vocabulary range
    vocab_size = len(tokenizer)
    preds = np.where(preds < vocab_size, preds, tokenizer.unk_token_id)

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)

    # Replace -100 in labels for decoding
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # Standard cleaning
    decoded_preds = [pred.strip() for pred in decoded_preds]
    decoded_labels = [[label.strip()] for label in decoded_labels]

    bleu = bleu_metric.compute(predictions=decoded_preds, references=decoded_labels)
    chrf = chrf_metric.compute(predictions=decoded_preds, references=decoded_labels)

    return {"bleu": bleu["score"], "chrf": chrf["score"]}

def preprocess_nllb_fixed(examples):
    model_inputs = tokenizer(examples["ro"], max_length=128, truncation=True)

    with tokenizer.as_target_tokenizer():
        labels = tokenizer(examples["rmy"], max_length=128, truncation=True)

    fixed_labels = []
    for label_seq in labels["input_ids"]:
        # Ensure the sequence starts with our verified RROMANI_ID
        # Filter out <s> (2) and <unk> (3) from the start
        cleaned_seq = [t for t in label_seq if t not in [tokenizer.bos_token_id, 3]]
        new_seq = [RROMANI_ID] + cleaned_seq
        fixed_labels.append([(l if l != tokenizer.pad_token_id else -100) for l in new_seq])

    model_inputs["labels"] = fixed_labels
    return model_inputs

train_dataset = train_ds.map(preprocess_nllb_fixed, batched=True, remove_columns=train_ds.column_names)
eval_dataset = eval_ds.map(preprocess_nllb_fixed, batched=True, remove_columns=eval_ds.column_names)

model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME, device_map="auto")
#model.config.forced_bos_token_id = tokenizer.convert_tokens_to_ids("rom_Latn")
model.config.forced_bos_token_id = 256204
model.config.decoder_start_token_id = 256204

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    task_type=TaskType.SEQ_2_SEQ_LM
)
model = get_peft_model(model, lora_config)


args = Seq2SeqTrainingArguments(
    output_dir="./nllb-ro-rom-v1",
    eval_strategy="epoch",
    learning_rate=2e-4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    weight_decay=0.01,
    num_train_epochs=5,
    predict_with_generate=True,
    fp16=True,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=tokenizer,
    data_collator=DataCollatorForSeq2Seq(tokenizer, model=model),
    compute_metrics=compute_metrics,
)

print("--- Initializing Baseline Evaluation ---")
baseline_metrics = trainer.evaluate()

print("\nBaseline Results:")
print(f"BLEU: {baseline_metrics['eval_bleu']:.2f}")
print(f"chrF: {baseline_metrics['eval_chrf']:.2f}")

#trainer.train()

Map:   0%|          | 0/9341 [00:00<?, ? examples/s]C:\Users\rober\anaconda3\envs\model_fine-tuning-v2\lib\site-packages\transformers\tokenization_utils_base.py:4034: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(
Map: 100%|██████████| 1038/1038 [00:00<00:00, 1766.80 examples/s]
C:\Users\rober\AppData\Local\Temp\ipykernel_51136\3495827310.py:48: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(
The model is already on multiple devices. Skipping the move to device specified in `args`.
C:\Users\rober\anaconda3\envs\model_fine-tuning-v2\lib\site-packages\transformers\generation\utils.py:1733: UserWarning: You 

--- Initializing Baseline Evaluation ---



Baseline Results:
BLEU: 0.00
chrF: 0.48


In [50]:
args = Seq2SeqTrainingArguments(
    output_dir="./nllb-rom-ron-results",
    # Evaluation Strategy
    eval_strategy="epoch",      # Run eval after each full pass
    save_strategy="epoch",            # Save a checkpoint so you don't lose progress
    logging_steps=50,                 # Log training loss every 50 steps

    # Hyperparameters for 8k pairs
    learning_rate=1e-4,               # Stable for LoRA
    per_device_train_batch_size=16,
    gradient_accumulation_steps=4,    # Effective batch size 64
    num_train_epochs=5,               # 5 passes is usually the sweet spot

    # Hardware/Speed
    fp16=True,                        # Keep enabled if your GPU supports it
    predict_with_generate=True,       # CRITICAL for BLEU/chrF calculation
)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset, # Map your preprocessed data here
    eval_dataset=eval_dataset,
    tokenizer=tokenizer,
    data_collator=DataCollatorForSeq2Seq(tokenizer, model=model, padding=True, label_pad_token_id=-100),
    compute_metrics=compute_metrics,
)

# Start the process
trainer.train()

C:\Users\rober\AppData\Local\Temp\ipykernel_33456\1254024100.py:19: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(
The model is already on multiple devices. Skipping the move to device specified in `args`.


Epoch,Training Loss,Validation Loss,Bleu,Chrf
1,7.450000,6.595549,0.016426,2.694087
2,6.042400,5.589825,0.075011,5.287942
3,5.649100,5.336489,0.189147,6.894103
4,5.467900,5.220985,0.234275,7.798498
5,5.406700,5.184924,0.279480,8.623715


TrainOutput(global_step=730, training_loss=6.054630687138806, metrics={'train_runtime': 4555.0088, 'train_samples_per_second': 10.254, 'train_steps_per_second': 0.16, 'total_flos': 6879264787636224.0, 'train_loss': 6.054630687138806, 'epoch': 5.0})

In [22]:
sample = train_dataset[0]
# Grab the first non -100 ID
first_id = [l for l in sample['labels'] if l != -100][0]

print(f"Target Language ID: {RROMANI_ID}")
print(f"First Label ID in Dataset: {first_id}")
print(f"Decoded: {tokenizer.decode([first_id])}")

Target Language ID: 256204
First Label ID in Dataset: 256204
Decoded: rom_Latn


In [23]:
# Force the model configuration to use the verified ID
model.config.forced_bos_token_id = 256204
model.config.decoder_start_token_id = 256204

# Double check that your labels in the dataset are also using this ID
sample = train_dataset[0]
first_label_id = [l for l in sample['labels'] if l != -100][0]
print(f"Dataset Verification: {first_label_id} == 256204")

Dataset Verification: 256204 == 256204


In [24]:
import pandas as pd

# Take 10 random samples from evaluation set
test_subset = eval_ds.select(range(10))

# Get predictions from the UNTRAINED model
raw_preds = trainer.predict(test_dataset=test_subset.map(preprocess_nllb_fixed, batched=True))

# Decode the predictions into actual text
decoded_preds = tokenizer.batch_decode(raw_preds.predictions, skip_special_tokens=True)
baseline_df = pd.DataFrame({
    "Romanian": test_subset["ro"],
    "Rromani_Reference": test_subset["rmy"],
    "Baseline_Prediction": decoded_preds
})

baseline_df.to_csv("baseline_translations.csv", index=False)
print("Baseline translations saved to baseline_translations.csv")

Map: 100%|██████████| 10/10 [00:00<00:00, 711.07 examples/s]


Baseline translations saved to baseline_translations.csv


In [34]:
import torch

def translate_romanian_to_rromani(text, model, tokenizer):
    model.eval()
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    # Generate with the forced Rromani tag
    with torch.no_grad():
        generated_tokens = model.generate(
            **inputs,
            forced_bos_token_id=256204,
            max_length=128,
            num_beams=5,
            no_repeat_ngram_size=3
        )

    return tokenizer.decode(generated_tokens[0], skip_special_tokens=True)

test_verse = "Teofile, în cea dintâi carte a mea, am vorbit despre tot ce a început Isus să facă şi să înveţe pe oameni"
print(f"RO: {test_verse}")
print(f"RM: {translate_romanian_to_rromani(test_verse, model, tokenizer)}")

RO: Teofile, în cea dintâi carte a mea, am vorbit despre tot ce a început Isus să facă şi să înveţe pe oameni
RM: O Teofilo, ai phendǎs len len lenqe k‐o le manushqe, phendevǎs savorǎs kaj O Jesus phende haj and‐o manushke.


## Load the model and evaluate

Baseline Results:
- BLEU: 0.00
- chrF: 0.48

Final Results:
- BLEU: 0.37
- chrF: 10.13

In [31]:
from transformers import AutoModelForSeq2SeqLM, NllbTokenizerFast
from peft import PeftModel, PeftConfig

# 1. Load config to get the base model path
adapter_path = "./nllb-rom-ron-results/checkpoint-730"
config = PeftConfig.from_pretrained(adapter_path)

# 2. FORCE use NllbTokenizerFast and explicit language tags
tokenizer = NllbTokenizerFast.from_pretrained(
    config.base_model_name_or_path,
    src_lang="ron_Latn",
    tgt_lang="rom_Latn",
    additional_special_tokens=["rom_Latn", "ron_Latn"]
)

# 3. Load base model and adapters
base_model = AutoModelForSeq2SeqLM.from_pretrained(
    config.base_model_name_or_path,
    device_map="auto"
)
model = PeftModel.from_pretrained(base_model, adapter_path)

# 4. Verify the ID matches your training
RROMANI_ID = tokenizer.convert_tokens_to_ids("rom_Latn")
print(f"Verified ID for inference: {RROMANI_ID}") # Should be 256204

Verified ID for inference: 256204


In [26]:
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, DataCollatorForSeq2Seq
import numpy as np

eval_args = Seq2SeqTrainingArguments(
    output_dir="./eval_results",
    predict_with_generate=True,
    generation_max_length=128,
    per_device_eval_batch_size=16,
    fp16=True
)

eval_trainer = Seq2SeqTrainer(
    model=model,
    args=eval_args,
    train_dataset=None,
    eval_dataset=eval_dataset,
    tokenizer=tokenizer,
    data_collator=DataCollatorForSeq2Seq(tokenizer, model=model, padding=True, label_pad_token_id=-100),
    compute_metrics=compute_metrics,
)

print("--- Running Final Evaluation ---")
results = eval_trainer.evaluate()

print(f"\nFinal Results:")
print(f"BLEU: {results['eval_bleu']:.2f}")
print(f"chrF: {results['eval_chrf']:.2f}")

C:\Users\rober\AppData\Local\Temp\ipykernel_51136\1677808190.py:12: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  eval_trainer = Seq2SeqTrainer(
The model is already on multiple devices. Skipping the move to device specified in `args`.


--- Running Final Evaluation ---



Final Results:
BLEU: 0.41
chrF: 9.77


## Continue training - stage 2

In [32]:
from transformers import EarlyStoppingCallback

# 1. Load config to get the base model path
adapter_path = "./nllb-rom-ron-results/checkpoint-730"
config = PeftConfig.from_pretrained(adapter_path)

# 1. Load the base model as usual
base_model = AutoModelForSeq2SeqLM.from_pretrained(
    config.base_model_name_or_path,
    device_map="auto"
)

# 2. LOAD ADAPTERS FOR TRAINING (is_trainable=True)
# This is the line that fixes the grad_fn error
model = PeftModel.from_pretrained(
    base_model,
    adapter_path,
    is_trainable=True
)

# 3. Double-check that you have trainable parameters
model.print_trainable_parameters()

# 1. Update training arguments for fine-tuning refinement
refined_args = Seq2SeqTrainingArguments(
    output_dir="./nllb-ro-rom-v2",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-5,               # Lower learning rate for Stage 2
    per_device_train_batch_size=16,
    gradient_accumulation_steps=4,
    num_train_epochs=10,              # Extended duration
    predict_with_generate=True,
    fp16=True,
    load_best_model_at_end=True,      # Keep the version with the lowest loss
    metric_for_best_model="chrf",      # chrF is more stable for low-resource
)

# 2. Re-initialize the trainer with the loaded model
trainer = Seq2SeqTrainer(
    model=model,
    args=refined_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=tokenizer,
    data_collator=DataCollatorForSeq2Seq(tokenizer, model=model, padding=True, label_pad_token_id=-100),
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

# 3. Resume training
trainer.train()

C:\Users\rober\AppData\Local\Temp\ipykernel_51136\2660647732.py:40: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(
The model is already on multiple devices. Skipping the move to device specified in `args`.


trainable params: 2,359,296 || all params: 617,433,088 || trainable%: 0.3821


Epoch,Training Loss,Validation Loss,Bleu,Chrf
1,No log,5.053458,0.373512,9.979418
2,No log,4.941330,0.473178,10.496461
3,No log,4.844390,0.623002,11.449291
4,5.228400,4.766767,0.646309,11.450352
5,5.228400,4.700205,0.711224,11.740427
6,5.228400,4.652506,0.692882,11.800680
7,4.968700,4.613836,0.737818,11.924806
8,4.968700,4.589455,0.791182,12.376783
9,4.968700,4.575022,0.818222,12.416549
10,4.968700,4.569401,0.821597,12.475068


TrainOutput(global_step=1460, training_loss=5.0248242521939215, metrics={'train_runtime': 8404.5758, 'train_samples_per_second': 11.114, 'train_steps_per_second': 0.174, 'total_flos': 1.3745727587868672e+16, 'train_loss': 5.0248242521939215, 'epoch': 10.0})

In [36]:
test_verse = "Toţi împreună erau nelipsiţi de la Templu în fiecare zi, frângeau pâinea acasă şi luau hrana cu bucurie şi curăţie de inimă."
correct_verse_translation = "Ande fiesavo ges arakhenas pen savorre khethanes k‑o Templo, phagenas o manro khere penθe haj xanas lośale haj uźe ileça."
print(f"RO: {test_verse}")
print(f"RM: {translate_romanian_to_rromani(test_verse, model, tokenizer)}")
print(f"Correct translation: {correct_verse_translation}")

RO: Toţi împreună erau nelipsiţi de la Templu în fiecare zi, frângeau pâinea acasă şi luau hrana cu bucurie şi curăţie de inimă.
RM: Antunchi kai zhanas le Devlesko, kai zhenas le manush, ai zhanen le pan, ai le manishas le zhanesko ai le zhenesko.
Correct translation: Ande fiesavo ges arakhenas pen savorre khethanes k‑o Templo, phagenas o manro khere penθe haj xanas lośale haj uźe ileça.
